In [1]:
import sys
sys.path.insert(0, '../src')

import jax
import jax.numpy as jnp
import haiku as hk 
import numpy as np 
import matplotlib.pyplot as plt 

KEY = jax.random.PRNGKey(42)

In [2]:
def mlp_fn(x): 
    x = hk.Linear(8, name="h0")(x); x = jax.nn.relu(x)
    x = hk.Linear(6, name="h1")(x); x = jax.nn.relu(x) 
    x = hk.Linear(3, name="out")(x); 
    return x 

mlp = hk.transform(mlp_fn)
params = mlp.init(KEY, jnp.zeros(4))
print(jax.tree.map(jnp.shape, params))

{'h0': {'b': (8,), 'w': (4, 8)}, 'h1': {'b': (6,), 'w': (8, 6)}, 'out': {'b': (3,), 'w': (6, 3)}}


In [3]:
x = jnp.array([1,2,3,4])

y = mlp.apply(params=params, x=x, rng=KEY)
print(y, y.shape)

[-1.1535357   1.3178382   0.78587174] (3,)


In [5]:
batch_num = 5 
X = jax.random.normal(KEY, shape=(batch_num, 4))
y = mlp.apply(params=params, x=X, rng=KEY)
print(y, y.shape)

[[-0.145268    0.17348291  0.05262177]
 [-0.04999519  0.18273893 -0.19529088]
 [-0.3115807   0.348335    0.08198567]
 [ 0.00773263  0.17641188  0.14812236]
 [ 0.17417528  0.08997704 -0.21727452]] (5, 3)


In [4]:
print("hello")

hello


# Forward and Caching h_l and a_l

In [18]:
def mlp_fn(x): 
    x = hk.Linear(8, name="h0")(x); x = jax.nn.relu(x)
    x = hk.Linear(6, name="h1")(x); x = jax.nn.relu(x) 
    x = hk.Linear(3, name="out")(x); 
    return x 

mlp = hk.transform(mlp_fn)
params = mlp.init(KEY, jnp.zeros(4))
print(jax.tree.map(jnp.shape, params))

batch_num = 2
X = jax.random.normal(KEY, shape=(batch_num, 4))
y = mlp.apply(params=params, x=X, rng=KEY)
print(y, y.shape)

{'h0': {'b': (8,), 'w': (4, 8)}, 'h1': {'b': (6,), 'w': (8, 6)}, 'out': {'b': (3,), 'w': (6, 3)}}
[[-0.145268    0.17348291  0.05262177]
 [-0.04999519  0.18273893 -0.19529088]] (2, 3)


In [15]:
def linear_pre_activation(params, h): 
    """Pre-activation of dense layer. "a_l" in the Hessian approximation paper.
    """
    W, b = params 
    return h @ W + b 

def _path_get(d, path): 
    """Walk a tuple of keys through a nested dict. 
    """
    for k in path: 
        d = d[k]
    return d 

def haiku_to_layers(params, network_architecture): 
    """Converts Haiku params and network architecture into a list of layers.
    """
    out = []
    for path, fn, activation in network_architecture: 
        sub = _path_get(params, path)
        out.append((fn, (sub["w"], sub["b"])))
    return out

In [16]:
network_architecture = [
    (("h0",), linear_pre_activation, "relu"), 
    (("h1",), linear_pre_activation, "relu"), 
    (("out",), linear_pre_activation, "linear")
]

layers = haiku_to_layers(params, network_architecture)
print(layers)

[(<function linear_pre_activation at 0x12bf76700>, (Array([[ 0.28742772,  0.37732333, -0.42767754, -0.3012351 , -0.5668951 ,
        -0.39261195, -0.22551541, -0.57879186],
       [-0.08433013, -0.02346802, -0.1960115 , -0.18741709,  0.6046915 ,
         0.10280242,  0.19594993, -0.13801369],
       [-0.45318288,  0.21375208, -0.20956972, -0.43633762, -0.12951128,
         0.32274482, -0.23069333,  0.24149157],
       [-0.06818055,  0.0396781 ,  0.349011  ,  0.57982904,  0.38018772,
        -0.5179687 , -0.32220033, -0.7066074 ]], dtype=float32), Array([0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32))), (<function linear_pre_activation at 0x12bf76700>, (Array([[-0.0711212 , -0.4426407 , -0.01518687, -0.3797069 ,  0.5839985 ,
        -0.16041183],
       [ 0.21717641,  0.22604269,  0.6974698 , -0.29130107,  0.42551136,
        -0.17667076],
       [ 0.16940804,  0.3440377 , -0.05589499, -0.15584637,  0.18745047,
        -0.5842133 ],
       [ 0.13209376, -0.30280644, -0.09498401,  0.073

In [ ]:
def forward_with_cache(layers, x): 
    """Run the layer list forward, caching the activations 'h' 
    and pre-activations 'a' per weight layer.

    Returns (output, cache) 
        - output: pre-activation of the last weight (head is linear).
        - cache: list of (h_in, a) tuples, one per weight layer.
    """
    h = x 
    cache = [] 
    last_weight = len(layers) - 1 
    out = None 

    for l, (fn, params_l) in enumerate(layers): 
        a = fn(params_l, h)
        cache.append((h,a))
        if l == last_weight: 
            out = a 
        else: 
            h = jax.nn.relu(a)

    return out, cache  

In [24]:
out, cache = forward_with_cache(layers, X)

print("input:", X)
print("output from hk.apply:", y)
print("output from forward_with_cache:", out)
print("cache:", cache); print("cache length:", len(cache))

input: [[-0.02830462  0.46713185  0.29570296  0.15354592]
 [-0.12403282  0.21692315 -1.440879    0.7558599 ]]
output from hk.apply: [[-0.145268    0.17348291  0.05262177]
 [-0.04999519  0.18273893 -0.19529088]]
output from forward_with_cache: [[-0.145268    0.17348291  0.05262177]
 [-0.04999519  0.18273893 -0.19529088]]
cache: [(Array([[-0.02830462,  0.46713185,  0.29570296,  0.15354592],
       [-0.12403282,  0.21692315, -1.440879  ,  0.7558599 ]],      dtype=float32), Array([[-0.19200519,  0.04765688, -0.08783913, -0.1190181 ,  0.31859583,
         0.07503963, -0.01977167, -0.08517502],
       [ 0.5475031 , -0.32989103,  0.57629466,  1.0636872 ,  0.6754639 ,
        -0.785551  ,  0.15934025, -0.8402055 ]], dtype=float32)), (Array([[0.        , 0.04765688, 0.        , 0.        , 0.31859583,
        0.07503963, 0.        , 0.        ],
       [0.5475031 , 0.        , 0.57629466, 1.0636872 , 0.6754639 ,
        0.        , 0.15934025, 0.        ]], dtype=float32), Array([[ 0.06166058, 